# 05 — Exercises: from a sales table to a model-ready one

**Block 2 — Exploratory Data Analysis · Lesson 05 · Homework**

The chain's buyer places next week's order for every store and product, and the commercial
director wants a model to do it better than "last week plus a feeling". Lesson 07 trains that
model. **Your job is to build the table it will learn from** — the same pipeline as the lesson
notebook, end to end, on the half-year of sales you explored in Lesson 04's exercises.

The five tasks chain: each one adds columns to the table the previous one produced.

| # | Task | What you practise | Lesson § | Time |
|---|---|---|---|---|
| 1 | Last week's sales as a feature | `sort_values`, `groupby().shift`, `transform` ratios | 1–2 | 15 min |
| 2 | Calendar features | `.dt` parts, elapsed time, sine/cosine | 3 | 10 min |
| 3 | Categories into numbers | one-hot, frequency and boolean encoding | 4 | 15 min |
| 4 | Log and scale | `np.log1p`, `StandardScaler` | 5 | 15 min |
| 5 | Which features, how many dimensions | correlation with the target, redundancy, `PCA` | 6–7 | 20 min |

### How to work

- Fill in every cell marked **`# TODO`**. The cell right after it checks your answer and
  prints `✅ Exercise N passed`, or an error message telling you what is still wrong.
- **Do not edit the check cells**, and do not edit the cells marked *given*. **Do not modify
  `sales`** either: the checks recompute every answer from it.
- Where a task says "leave it NaN", leave it NaN. Filling a missing lag with 0 tells a model
  the store sold nothing — the checks will say so.
- Two tasks ask for one **plotly** figure. The check only looks at its *kind*; whether it is a
  *good* chart is up to you.
- Stuck? The lesson notebook has the matching section, and every task names it.

## 0. Setup — *given, run it and move on*

In [ ]:
import sys

import numpy as np
import pandas as pd
import plotly.express as px  # noqa: F401 -- used in your answers
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.decomposition import PCA

if "google.colab" in sys.modules:
    pio.renderers.default = "colab"
else:
    # Interactive charts in JupyterLab / VS Code, plus an HTML fallback that loads plotly.js from a CDN.
    pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_white"

SEED = 7
rng = np.random.default_rng(SEED)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:,.2f}".format)


def passed(n):
    print(f"✅ Exercise {n} passed")


print("pandas", pd.__version__)

## 1. The sales table — *given*

The same chain as Lesson 04's exercises: eight stores in four regions, fifteen products in
five categories, twenty-six weeks, one row per store, product and week — plus two raw columns
you did not need then: when each store opened, and the label printed on each promotion. The rows
come in no particular order, as they would from a warehouse export.

**In your Assignment Brief this cell disappears** and you load your own cleaned table instead.

In [ ]:
# ---------------------------------------------------------------- the chain's weekly sales
STORES = pd.DataFrame(
    {
        "store_id": [f"S{i:02d}" for i in range(1, 9)],
        "region": ["North", "North", "South", "South", "East", "East", "West", "West"],
        "size": rng.lognormal(0, 0.25, 8),  # bigger stores sell more of everything
    }
)
CATEGORIES = {  # (base price €, base weekly units)
    "Beverages": (1.6, 120),
    "Snacks": (2.3, 90),
    "Dairy": (1.9, 100),
    "Household": (6.5, 35),
    "Personal care": (8.9, 25),
}
PRODUCTS = pd.DataFrame(
    [
        (
            f"P{k:02d}",
            cat,
            round(price * rng.uniform(0.8, 1.25), 2),
            units * rng.uniform(0.7, 1.3),
        )
        for k, (cat, (price, units)) in enumerate(
            [(c, v) for c, v in CATEGORIES.items() for _ in range(3)], start=1
        )
    ],
    columns=["product_id", "category", "list_price", "base_units"],
)
WEEKS = pd.date_range("2026-01-05", periods=26, freq="W-MON")

grid = STORES.merge(PRODUCTS, how="cross").merge(
    pd.DataFrame({"week": WEEKS, "t": range(len(WEEKS))}), how="cross"
)
n = len(grid)
promo = rng.random(n) < 0.15
discount = np.where(promo, rng.uniform(0.20, 0.30, n), rng.uniform(0.0, 0.05, n))
season = (
    1 + 0.20 * np.sin(2 * np.pi * grid["t"] / 26) + 0.008 * grid["t"]
)  # a wave and a slow rise
expected = grid["base_units"] * grid["size"] * season * np.where(promo, 1.45, 1.0)
units = rng.poisson(expected)
bulk = rng.choice(n, 12, replace=False)  # a dozen bulk orders from local businesses
units[bulk] = units[bulk] * 8

sales = pd.DataFrame(
    {
        "week": grid["week"],
        "store_id": grid["store_id"],
        "region": grid["region"],
        "product_id": grid["product_id"],
        "category": grid["category"],
        "price": np.round(grid["list_price"] * (1 - discount), 2),
        "discount_pct": np.round(discount * 100, 1),
        "promo": promo,
        "units": units,
    }
)
sales["revenue"] = np.round(sales["units"] * sales["price"], 2)

# ---------------------------------------------------------------- two raw columns, new this lesson
STORES["store_open_date"] = pd.Timestamp("2026-01-05") - pd.to_timedelta(
    rng.integers(200, 4000, 8), unit="D"
)
sales = sales.merge(STORES[["store_id", "store_open_date"]], on="store_id", how="left")
LABELS = ["2x1", "-25%", "3 for 2", "Loyalty -20%", "Multibuy"]
sales["promo_label"] = np.where(sales["promo"], rng.choice(LABELS, n), "none")
sales = sales.sample(frac=1, random_state=SEED).reset_index(
    drop=True
)  # a warehouse export is in no useful order

print(sales.shape)
sales.head()

Three names the whole notebook shares — *given*. `KEYS` identifies a row; `GROUP` is the
series a lag runs along; `FEATURES` is the list of numeric columns tasks 4 and 5 work with
(you will have built every one of them by the end of task 3).

In [ ]:
KEYS = ["store_id", "product_id", "week"]
GROUP = ["store_id", "product_id"]
FEATURES = [
    "price",
    "discount_pct",
    "price_index",
    "store_share",
    "units_lag1",
    "log_units_lag1",
    "log_revenue",
    "store_age_weeks",
    "sin_week",
    "cos_week",
    "promo_label_freq",
]

---

## Exercise 1 — Last week's sales as a feature

*Lesson sections 1–2 · 15 minutes*

Build `feat`, a copy of `sales` **sorted by `KEYS`** (and with the index reset), with four new
columns:

1. `units_lag1` — the `units` of the **previous week** of the same store × product.
2. `units_next` — the `units` of the **following week** of the same store × product. This is
   the target.
3. `store_share` — this row's `revenue` as a share of the total revenue of the **same store
   in the same week**.
4. `price_index` — this row's `price` divided by the **median price of the same product**
   over the whole table.

The first week of every store × product has no previous week, and the last has no next week:
**leave those NaN.**

> **Hint.** `g = feat.groupby(GROUP)` once, then `g["units"].shift(1)` and `.shift(-1)`. For
> the ratios, divide by a `groupby(...)[...].transform("sum")` or `transform("median")`.

In [ ]:
# TODO: sorted copy of sales, index reset
feat = ...

# TODO: lag, target, two ratios
feat["units_lag1"] = ...
feat["units_next"] = ...
feat["store_share"] = ...
feat["price_index"] = ...

feat[KEYS + ["units", "units_lag1", "units_next", "store_share", "price_index"]].head(8)

In [ ]:
# --- check 1 (do not edit) ---
assert len(sales) == 3_120 and {"units", "revenue", "price"} <= set(
    sales.columns
), "`sales` has been modified — re-run the given cells"
_t = sales.sort_values(KEYS).reset_index(drop=True).copy()
_g = _t.groupby(GROUP)
_t["units_lag1"], _t["units_next"] = _g["units"].shift(1), _g["units"].shift(-1)
_t["store_share"] = _t["revenue"] / _t.groupby(["store_id", "week"])[
    "revenue"
].transform("sum")
_t["price_index"] = _t["price"] / _t.groupby("product_id")["price"].transform("median")
_cols = ["units_lag1", "units_next", "store_share", "price_index"]
assert isinstance(feat, pd.DataFrame) and set(KEYS) <= set(
    feat.columns
), "feat must be a DataFrame that keeps the key columns"
for _c in _cols:
    assert _c in feat.columns, f"feat has no column `{_c}` — add it in the task cell"
    assert not (
        feat[_c].dtype == object and feat[_c].iloc[0] is Ellipsis
    ), f"feat['{_c}'] is still `...` — the TODO was left unfilled"
assert (
    len(feat) == 3_120
), f"feat must have 3,120 rows (one per store × product × week), got {len(feat):,}"
_m = _t[KEYS + _cols].merge(
    feat[KEYS + _cols], on=KEYS, how="inner", suffixes=("_truth", "")
)
assert (
    len(_m) == 3_120
), "feat does not hold the expected store × product × week rows — duplicated or missing keys, or the wrong rows kept"
_pairs = len(sales[GROUP].drop_duplicates())
assert (
    int(_m["units_lag1"].isna().sum()) == _pairs
), "units_lag1 must be NaN for exactly the first week of every store × product — leave it NaN, do not fill it"
assert (
    int(_m["units_next"].isna().sum()) == _pairs
), "units_next must be NaN for exactly the last week of every store × product — leave it NaN"
_msgs = {
    "units_lag1": "units_lag1 is not last week's units of the same store × product — sort by week and shift(1) within groupby(store_id, product_id)",
    "units_next": "units_next is not next week's units of the same store × product — shift(-1) within groupby(store_id, product_id)",
    "store_share": "store_share = revenue / the total revenue of the same store in the same week (transform('sum') over store_id and week)",
    "price_index": "price_index = price / the median price of the same product (transform('median') over product_id)",
}
for _c, _msg in _msgs.items():
    assert np.allclose(
        _m[_c].astype(float), _m[_c + "_truth"].astype(float), equal_nan=True
    ), _msg
passed(1)

---

## Exercise 2 — Calendar features

*Lesson section 3 · 10 minutes*

Add five columns to `feat`, all computed from `week` and `store_open_date`:

1. `month` — the month number.
2. `week_of_year` — the ISO week number, as an **integer**.
3. `store_age_weeks` — the number of **whole weeks** from `store_open_date` to `week`.
4. `sin_week` and `cos_week` — the cyclic encoding of `week_of_year` over a 52-week year:
   `sin(2π · week_of_year / 52)` and `cos(2π · week_of_year / 52)`.

> **Hint.** `feat["week"].dt.isocalendar().week.astype(int)`; a difference of two datetime
> columns has `.dt.days`.

In [ ]:
# TODO
feat["month"] = ...
feat["week_of_year"] = ...
feat["store_age_weeks"] = ...
feat["sin_week"] = ...
feat["cos_week"] = ...

feat[
    [
        "week",
        "store_id",
        "month",
        "week_of_year",
        "store_age_weeks",
        "sin_week",
        "cos_week",
    ]
].drop_duplicates("week").head()

In [ ]:
# --- check 2 (do not edit) ---
assert len(sales) == 3_120 and {"week", "store_open_date"} <= set(
    sales.columns
), "`sales` has been modified — re-run the given cells"
_t = sales.sort_values(KEYS).reset_index(drop=True).copy()
_t["month"] = _t["week"].dt.month
_t["week_of_year"] = _t["week"].dt.isocalendar().week.astype(int)
_t["store_age_weeks"] = (_t["week"] - _t["store_open_date"]).dt.days // 7
_t["sin_week"], _t["cos_week"] = (
    np.sin(2 * np.pi * _t["week_of_year"] / 52),
    np.cos(2 * np.pi * _t["week_of_year"] / 52),
)
_cols = ["month", "week_of_year", "store_age_weeks", "sin_week", "cos_week"]
assert isinstance(feat, pd.DataFrame) and set(KEYS) <= set(
    feat.columns
), "feat must be a DataFrame that keeps the key columns"
for _c in _cols:
    assert _c in feat.columns, f"feat has no column `{_c}` — add it in the task cell"
    assert not (
        feat[_c].dtype == object and feat[_c].iloc[0] is Ellipsis
    ), f"feat['{_c}'] is still `...` — the TODO was left unfilled"
assert len(feat) == 3_120, f"feat must have 3,120 rows, got {len(feat):,}"
_m = _t[KEYS + _cols].merge(
    feat[KEYS + _cols], on=KEYS, how="inner", suffixes=("_truth", "")
)
assert (
    len(_m) == 3_120
), "feat does not hold the expected store × product × week rows — duplicated or missing keys, or the wrong rows kept"
assert np.allclose(
    _m["month"].astype(float), _m["month_truth"].astype(float)
), "month is week.dt.month"
assert np.allclose(
    _m["week_of_year"].astype(float), _m["week_of_year_truth"].astype(float)
), "week_of_year is week.dt.isocalendar().week (as an integer)"
assert (
    _m["store_age_weeks"].min() > 0
), "store_age_weeks must be positive: every store opened before the first week of data"
assert np.allclose(
    _m["store_age_weeks"].astype(float), _m["store_age_weeks_truth"].astype(float)
), "store_age_weeks = whole weeks between store_open_date and week: (week - store_open_date).dt.days // 7"
assert np.allclose(
    _m["sin_week"].astype(float), _m["sin_week_truth"]
), "sin_week = sin(2π · week_of_year / 52)"
assert np.allclose(
    _m["cos_week"].astype(float), _m["cos_week_truth"]
), "cos_week = cos(2π · week_of_year / 52)"
passed(2)

---

## Exercise 3 — Categories into numbers

*Lesson section 4 · 15 minutes*

Build `encoded` from `feat`:

1. **One-hot encode** `region` and `category` with `pd.get_dummies(feat, columns=[...])`, as
   **integer** 0/1 columns. The two text columns disappear; `region_North`, `category_Dairy`,
   … appear.
2. Add `promo_label_freq`: each row's `promo_label` replaced by the **share of rows** that
   carry that label (frequency encoding).
3. Convert `promo` from `True`/`False` to `1`/`0`.

> **Hint.** `dtype=int` in `get_dummies`; `.map(feat["promo_label"].value_counts(normalize=True))`;
> `.astype(int)`.

In [ ]:
# TODO: one-hot region and category, as integers
encoded = ...

# TODO: frequency encoding of the promotion label, and the boolean as 0/1
encoded["promo_label_freq"] = ...
encoded["promo"] = ...

print(encoded.shape)
encoded.filter(regex="^(region_|category_|promo)").head()

In [ ]:
# --- check 3 (do not edit) ---
assert len(sales) == 3_120 and {"region", "category", "promo", "promo_label"} <= set(
    sales.columns
), "`sales` has been modified — re-run the given cells"
_t = sales.sort_values(KEYS).reset_index(drop=True).copy()
_t["promo_label_freq"] = _t["promo_label"].map(
    _t["promo_label"].value_counts(normalize=True)
)
_dummy_cols = sorted(
    f"{c}_{v}" for c in ["region", "category"] for v in sales[c].unique()
)
assert isinstance(encoded, pd.DataFrame) and set(KEYS) <= set(
    encoded.columns
), "encoded must be a DataFrame that keeps the key columns"
assert not (
    {"region", "category"} & set(encoded.columns)
), "encoded must not keep the text columns region and category — pd.get_dummies(feat, columns=[...]) replaces them"
_missing = [c for c in _dummy_cols if c not in encoded.columns]
assert not _missing, f"encoded is missing dummy columns {_missing} — one 0/1 column per level, named <column>_<level>"
for _c in ["promo_label_freq", "promo"]:
    assert _c in encoded.columns and not (
        encoded[_c].dtype == object and encoded[_c].iloc[0] is Ellipsis
    ), f"encoded['{_c}'] is missing or still `...`"
assert len(encoded) == 3_120, f"encoded must have 3,120 rows, got {len(encoded):,}"
_td = pd.get_dummies(
    _t[KEYS + ["region", "category"]], columns=["region", "category"], dtype=int
)
_m = _td.merge(
    encoded[KEYS + _dummy_cols], on=KEYS, how="inner", suffixes=("_truth", "")
)
assert (
    len(_m) == 3_120
), "encoded does not hold the expected store × product × week rows — duplicated or missing keys, or the wrong rows kept"
for _col in ["region", "category"]:
    _grp = [c for c in _dummy_cols if c.startswith(f"{_col}_")]
    assert (
        _m[_grp].astype(float).sum(axis=1) == 1
    ).all(), f"every row must have exactly one 1 across the {_col}_* columns"
for _c in _dummy_cols:
    assert np.allclose(
        _m[_c].astype(float), _m[_c + "_truth"].astype(float)
    ), f"{_c} does not match the original `{_c.split('_')[0]}` value"
_m = _t[KEYS + ["promo_label_freq"]].merge(
    encoded[KEYS + ["promo_label_freq"]], on=KEYS, suffixes=("_truth", "")
)
assert np.allclose(
    _m["promo_label_freq"].astype(float), _m["promo_label_freq_truth"]
), "promo_label_freq is the share of rows carrying that label: map(value_counts(normalize=True))"
assert pd.api.types.is_integer_dtype(encoded["promo"]) and set(
    encoded["promo"].unique()
) <= {0, 1}, "promo must be converted to 0/1 integers (astype(int))"
passed(3)

---

## Exercise 4 — Log and scale

*Lesson section 5 · 15 minutes*

1. Build `model_df`: the rows of `encoded` where **both** `units_lag1` and `units_next` are
   present (drop the others — a model cannot learn from a row without a target or without its
   main feature). Make it a copy.
2. Add `log_units_lag1` = `log1p(units_lag1)` and `log_revenue` = `log1p(revenue)`.
3. Fit `scaler`, a `StandardScaler`, on `model_df[FEATURES]`.
4. Build `X_scaled`: a DataFrame of `scaler.transform(model_df[FEATURES])` with the `FEATURES`
   as column names and **the index of `model_df`**.
5. Draw `fig4`: a **histogram** of `log_units_lag1`.

No train/test split in this homework — no model is fitted, so there is nothing to leak into.
The lesson notebook shows the version with the split; Lesson 07 always uses it.

In [ ]:
# TODO: rows with a lag and a target
model_df = ...

# TODO: the two log columns
model_df["log_units_lag1"] = ...
model_df["log_revenue"] = ...

# TODO: fit, transform, wrap in a DataFrame
scaler = ...
X_scaled = ...

# TODO: a histogram of log_units_lag1
fig4 = ...

print(model_df.shape, X_scaled.shape)
X_scaled.describe().loc[["mean", "std"]].round(2)

In [ ]:
# --- check 4 (do not edit) ---
assert len(sales) == 3_120 and {"units", "revenue", "price"} <= set(
    sales.columns
), "`sales` has been modified — re-run the given cells"
_t = sales.sort_values(KEYS).reset_index(drop=True).copy()
_g = _t.groupby(GROUP)
_t["units_lag1"], _t["units_next"] = _g["units"].shift(1), _g["units"].shift(-1)
_t["store_share"] = _t["revenue"] / _t.groupby(["store_id", "week"])[
    "revenue"
].transform("sum")
_t["price_index"] = _t["price"] / _t.groupby("product_id")["price"].transform("median")
_t["week_of_year"] = _t["week"].dt.isocalendar().week.astype(int)
_t["store_age_weeks"] = (_t["week"] - _t["store_open_date"]).dt.days // 7
_t["sin_week"], _t["cos_week"] = (
    np.sin(2 * np.pi * _t["week_of_year"] / 52),
    np.cos(2 * np.pi * _t["week_of_year"] / 52),
)
_t["promo_label_freq"] = _t["promo_label"].map(
    _t["promo_label"].value_counts(normalize=True)
)
_t = _t.dropna(subset=["units_lag1", "units_next"]).copy()
_t["log_units_lag1"], _t["log_revenue"] = (
    np.log1p(_t["units_lag1"]),
    np.log1p(_t["revenue"]),
)
_x = _t[FEATURES].astype(float)
_ts = pd.concat([_t[KEYS], (_x - _x.mean()) / _x.std(ddof=0)], axis=1)
_n = len(_t)
assert isinstance(model_df, pd.DataFrame) and set(KEYS) <= set(
    model_df.columns
), "model_df must be a DataFrame that keeps the key columns"
for _c in ["log_units_lag1", "log_revenue"]:
    assert _c in model_df.columns and not (
        model_df[_c].dtype == object and model_df[_c].iloc[0] is Ellipsis
    ), f"model_df['{_c}'] is missing or still `...`"
assert (
    len(model_df) == _n
), f"model_df must have {_n:,} rows (one per store × product × week), got {len(model_df):,}"
_m = _t[KEYS + ["log_units_lag1", "log_revenue"]].merge(
    model_df[KEYS + ["log_units_lag1", "log_revenue"]], on=KEYS, suffixes=("_truth", "")
)
assert (
    len(_m) == _n
), "model_df does not hold the expected store × product × week rows — duplicated or missing keys, or the wrong rows kept"
assert np.allclose(
    _m["log_units_lag1"].astype(float), _m["log_units_lag1_truth"]
), "log_units_lag1 = np.log1p(units_lag1) — and keep only the rows where units_lag1 and units_next are not NaN"
assert np.allclose(
    _m["log_revenue"].astype(float), _m["log_revenue_truth"]
), "log_revenue = np.log1p(revenue)"
_mean = getattr(scaler, "mean_", None)
assert (
    _mean is not None and len(_mean) == len(FEATURES)
), "scaler must be a StandardScaler fitted on model_df[FEATURES] (call .fit; it then has a mean_ of 11 values)"
assert np.allclose(
    np.sort(_mean), np.sort(_x.mean().to_numpy())
), "scaler was not fitted on model_df[FEATURES] — its means do not match those columns"
assert (
    isinstance(X_scaled, pd.DataFrame) and set(FEATURES) <= set(X_scaled.columns)
), "X_scaled must be a DataFrame with the FEATURES as columns (wrap scaler.transform(...) in pd.DataFrame)"
assert len(X_scaled) == _n and X_scaled.index.equals(
    model_df.index
), "X_scaled must keep model_df's index (pass index=model_df.index)"
assert (
    np.allclose(X_scaled[FEATURES].astype(float).mean(), 0, atol=1e-6)
    and np.allclose(X_scaled[FEATURES].astype(float).std(ddof=0), 1, atol=1e-6)
), "after StandardScaler every column has mean 0 and std 1 — X_scaled is not the scaler's output"
_m = _ts.merge(
    model_df[KEYS].join(X_scaled[FEATURES]), on=KEYS, suffixes=("_truth", "")
)
assert (
    len(_m) == _n
), "X_scaled rows do not line up with model_df — keep model_df's index"
for _c in FEATURES:
    assert np.allclose(
        _m[_c].astype(float), _m[_c + "_truth"]
    ), f"X_scaled['{_c}'] is not (x - mean) / std of model_df['{_c}'] — rows may be out of order"
assert (
    isinstance(fig4, go.Figure) and fig4.data
), "fig4 must be a plotly figure — use px.histogram"
assert (
    fig4.data[0].type == "histogram"
), f"fig4 is a {fig4.data[0].type} chart, not a histogram"
passed(4)

---

## Exercise 5 — Which features, how many dimensions

*Lesson sections 6–7 · 20 minutes*

1. `target_corr`: a Series with the **absolute** Pearson correlation of each of the
   `FEATURES` with `units_next` (on `model_df`), **sorted in descending order**.
2. `top3`: the list of the three feature names with the largest `|r|`.
3. `redundant_pairs`: a list of `(feature_a, feature_b)` tuples, one per pair of `FEATURES`
   whose mutual `|r|` is **0.95 or more** (each pair once).
4. `pca`: a `PCA()` fitted on `X_scaled` (all components).
5. `n_components_95`: the smallest number of components whose **cumulative** explained
   variance reaches 95%.
6. `fig5`: a **bar** chart of `target_corr`.

> **Hint.** `model_df[FEATURES].corrwith(model_df["units_next"])`. For the pairs, take the
> correlation matrix, keep its upper triangle with `np.triu`, `stack()` it and filter. For the
> count, `np.cumsum(pca.explained_variance_ratio_)` and the first position `>= 0.95`.

In [ ]:
# TODO: |r| with the target, descending, and the top three
target_corr = ...
top3 = ...

# TODO: pairs of features with |r| >= 0.95
redundant_pairs = ...

# TODO: PCA on the scaled matrix, and how many components keep 95%
pca = ...
n_components_95 = ...

# TODO: a bar chart of target_corr
fig5 = ...

print("top 3:", top3)
print("redundant:", redundant_pairs)
print("components for 95%:", n_components_95, "of", len(FEATURES))

In [ ]:
# --- check 5 (do not edit) ---
assert len(sales) == 3_120 and {"units", "revenue", "price"} <= set(
    sales.columns
), "`sales` has been modified — re-run the given cells"
_t = sales.sort_values(KEYS).reset_index(drop=True).copy()
_g = _t.groupby(GROUP)
_t["units_lag1"], _t["units_next"] = _g["units"].shift(1), _g["units"].shift(-1)
_t["store_share"] = _t["revenue"] / _t.groupby(["store_id", "week"])[
    "revenue"
].transform("sum")
_t["price_index"] = _t["price"] / _t.groupby("product_id")["price"].transform("median")
_t["week_of_year"] = _t["week"].dt.isocalendar().week.astype(int)
_t["store_age_weeks"] = (_t["week"] - _t["store_open_date"]).dt.days // 7
_t["sin_week"], _t["cos_week"] = (
    np.sin(2 * np.pi * _t["week_of_year"] / 52),
    np.cos(2 * np.pi * _t["week_of_year"] / 52),
)
_t["promo_label_freq"] = _t["promo_label"].map(
    _t["promo_label"].value_counts(normalize=True)
)
_t = _t.dropna(subset=["units_lag1", "units_next"]).copy()
_t["log_units_lag1"], _t["log_revenue"] = (
    np.log1p(_t["units_lag1"]),
    np.log1p(_t["revenue"]),
)
_x = _t[FEATURES].astype(float)
_tc = _x.corrwith(_t["units_next"].astype(float)).abs().sort_values(ascending=False)
assert isinstance(target_corr, pd.Series) and set(target_corr.index) == set(
    FEATURES
), "target_corr must be a Series indexed by the 11 FEATURES"
assert (
    target_corr.is_monotonic_decreasing
), "sort target_corr in descending order (sort_values(ascending=False))"
assert np.allclose(
    target_corr.loc[FEATURES].astype(float), _tc.loc[FEATURES]
), "target_corr is the ABSOLUTE Pearson correlation of each feature with units_next, on model_df"
assert (
    list(top3) == list(_tc.index[:3])
), "top3 is the list of the three feature names with the largest |r| — the first three of target_corr"
_corr = _x.corr().abs()
_upper = _corr.where(np.triu(np.ones(_corr.shape, dtype=bool), k=1))
_truth_pairs = {frozenset(p) for p in _upper.stack()[_upper.stack() >= 0.95].index}
assert isinstance(redundant_pairs, (list, tuple, set)) and all(
    len(tuple(p)) == 2 for p in redundant_pairs
), "redundant_pairs must be a list of (feature_a, feature_b) pairs"
assert (
    {frozenset(p) for p in redundant_pairs} == _truth_pairs
), f"redundant_pairs must list every pair of FEATURES with |r| >= 0.95 exactly once (there are {len(_truth_pairs)})"
_evr = getattr(pca, "explained_variance_ratio_", None)
assert _evr is not None and len(_evr) == len(
    FEATURES
), "pca must be a PCA() fitted on X_scaled (all 11 components)"
_truth_evr = (
    PCA().fit(((_x - _x.mean()) / _x.std(ddof=0)).to_numpy()).explained_variance_ratio_
)
assert np.allclose(
    _evr, _truth_evr, atol=1e-6
), "pca does not match a PCA fitted on X_scaled — fit it on the scaled matrix, not on model_df"
assert (
    int(n_components_95) == int(np.argmax(np.cumsum(_truth_evr) >= 0.95) + 1)
), "n_components_95 is the smallest number of components whose cumulative explained variance reaches 0.95 — count from 1"
assert (
    isinstance(fig5, go.Figure) and fig5.data
), "fig5 must be a plotly figure — use px.bar"
assert fig5.data[0].type == "bar", f"fig5 is a {fig5.data[0].type} chart, not a bar"
passed(5)

---

## Done

A sorted table, four lags and ratios, five calendar columns, nine dummies, two logs, one
scaler, and a short list of what to keep: that is a **feature table**, and it is what a model
learns from. Nothing in it uses next week's numbers — the one rule that matters.

| What you did | Where it shows up again |
|---|---|
| Built lags and the target inside each store × product | Lesson 07 — the first model predicts `units_next` from exactly this table |
| Encoded categories without touching the target | Lesson 09 — target encoding done safely, on a separate fold |
| Scaled the features, and knew why | Lesson 07 — the models that need it, and the trees that do not |
| Dropped a redundant pair and compressed the rest | Lesson 09 — interpretability: which feature drove the number |